# LeetCode 854 — K-Similar Strings bằng Python

- Link: https://leetcode.com/problems/k-similar-strings/
- Độ khó: **Hard**
- Chủ đề: BFS trên không gian trạng thái chuỗi, Cắt tỉa nhánh, Backtracking + memo, A* với heuristic chấp nhận được, Phân rã chu trình hoán vị

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kSimilarity(self, s1: str, s2: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Hai chuỗi `s1`, `s2` là anagram của nhau. Mỗi bước, ta được **đổi chỗ hai ký tự** trong `s1`. Hỏi số lần đổi chỗ **ít nhất** để `s1` thành `s2`. (Đề gọi số đó là `k`, và nói hai chuỗi "k-similar".)

Thoạt nhìn giống bài sắp xếp bằng hoán đổi — với các phần tử **khác nhau**, đáp án là `n − (số chu trình)` của hoán vị. Nhưng ở đây ký tự **lặp lại** (chỉ có 6 chữ `a..f`, độ dài tới 20), nên có nhiều cách ghép vị trí và bài trở nên khó (bài tổng quát là NP-khó). Với `n <= 20`, ta dùng **tìm kiếm có cắt tỉa thông minh**: BFS chỉ sửa **vị trí sai đầu tiên**, hoặc DFS/A* với cận dưới.

Sau notebook này bạn sẽ:

1. Mô hình hoá thành BFS trên đồ thị chuỗi (mỗi cạnh = một lần đổi chỗ).
2. Hiểu và chứng minh cắt tỉa "chỉ sửa vị trí sai đầu tiên, chỉ đổi với `j` mà `s[j]` đúng chữ cần và `s[j]` đang sai".
3. Viết backtracking có memo và **cận dưới** `ceil(sai / 2)` để cắt nhánh.
4. Cài A* với heuristic chấp nhận được.
5. Liên hệ với **phân rã chu trình**: ưu tiên đổi chỗ sửa được **hai** vị trí cùng lúc.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Hai chuỗi `s1`, `s2` là **k-similar** nếu có thể đổi chỗ hai ký tự trong `s1` đúng `k` lần để được `s2`. Cho `s1`, `s2` là anagram của nhau, trả về `k` **nhỏ nhất**.

**Ví dụ 1**

```text
Input: s1 = "ab", s2 = "ba"
Output: 1
Giải thích: đổi chỗ 'a' và 'b'.
```

**Ví dụ 2**

```text
Input: s1 = "abc", s2 = "bca"
Output: 2
Giải thích: "abc" → "bac" → "bca".
```

**Ràng buộc**

- `1 <= s1.length <= 20`
- `s2.length == s1.length`
- `s1`, `s2` chỉ gồm các chữ trong `{'a','b','c','d','e','f'}`.
- `s2` là một anagram của `s1`.

In [1]:
from typing import List

EXAMPLES = [
    ("ab", "ba", 1),
    ("abc", "bca", 2),
    ("abc", "abc", 0),
    ("aabc", "abca", 2),
    ("abac", "baca", 2),
    ("abcdef", "badcfe", 3),   # ba chu trình độ dài 2
]
for a, b, exp in EXAMPLES:
    print(a, "->", b, ": kỳ vọng", exp)

ab -> ba : kỳ vọng 1
abc -> bca : kỳ vọng 2
abc -> abc : kỳ vọng 0
aabc -> abca : kỳ vọng 2
abac -> baca : kỳ vọng 2
abcdef -> badcfe : kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Chuỗi bất biến — đổi chỗ bằng list

Chuỗi Python không sửa tại chỗ được. Để đổi chỗ `i`, `j`: chuyển sang `list`, đổi, rồi `''.join`.

In [2]:
def swap(s, i, j):
    t = list(s)
    t[i], t[j] = t[j], t[i]
    return ''.join(t)

print(swap("abcd", 0, 3))

dbca


### 3.2. BFS trên chuỗi

Đỉnh là chuỗi, `dist` lưu trong `dict`. Lần đầu gặp `s2` là số bước ít nhất.

In [3]:
from collections import deque

def bfs_generic(start, goal, neighbors):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        if u == goal:
            return dist[u]
        for v in neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)

print(bfs_generic("ab", "ba", lambda s: [swap(s, 0, 1)]))

1


### 3.3. Hoán vị và chu trình (khi phần tử khác nhau)

Nếu mọi ký tự khác nhau, mỗi vị trí `i` có "đích" duy nhất `p[i]`. Hoán vị `p` phân rã thành các chu trình; số lần đổi chỗ tối thiểu = `n − số chu trình` (mỗi chu trình độ dài `L` cần `L − 1` lần đổi).

In [4]:
def cycles_of_perm(p):
    n = len(p); seen = [False] * n; cyc = []
    for i in range(n):
        if not seen[i]:
            c = []
            while not seen[i]:
                seen[i] = True; c.append(i); i = p[i]
            cyc.append(c)
    return cyc

s1, s2 = "abcdef", "bcaefd"
p = [s2.index(ch) for ch in s1]            # ký tự s1[i] phải tới vị trí p[i]
print(p, cycles_of_perm(p), "→ số lần đổi:", len(s1) - len(cycles_of_perm(p)))

[2, 0, 1, 5, 3, 4] [[0, 2, 1], [3, 5, 4]] → số lần đổi: 4


### 3.4. Heap cho A*

`heapq` lấy phần tử có `f = g + h` nhỏ nhất.

In [5]:
import heapq
pq = []
for item in [(3, "x"), (1, "y"), (2, "z")]:
    heapq.heappush(pq, item)
print([heapq.heappop(pq) for _ in range(3)])

[(1, 'y'), (2, 'z'), (3, 'x')]


### 3.5. Memo hoá bằng `dict`

DFS trả về số bước tối thiểu từ chuỗi hiện tại tới `s2`; lưu theo khoá chuỗi để không tính lại.

In [6]:
memo = {}
def demo(s):
    if s in memo:
        return memo[s]
    memo[s] = len(s)
    return memo[s]
print(demo("abc"), memo)

3 {'abc': 3}


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Một lần đổi chỗ sửa được tối đa 2 vị trí

Gọi `mis` = số vị trí `i` mà `s1[i] != s2[i]`. Một lần đổi chỗ thay đổi đúng 2 vị trí → `mis` giảm tối đa 2. Vậy **cận dưới**: `k >= ceil(mis / 2)`.

### 4.2. Tính tay `s1 = "abc"`, `s2 = "bca"`

`mis = 3` → `k >= 2`. Thử: vị trí 0 cần `'b'`, `'b'` đang ở vị trí 1 → đổi 0↔1: `"bac"`. Vị trí 1 cần `'c'`, `'c'` ở vị trí 2 → đổi 1↔2: `"bca"` ✔. `k = 2`.

### 4.3. Đổi chỗ "tốt" và "xấu"

Đổi `i ↔ j` sao cho cả hai vị trí đều đúng (`s[j] == s2[i]` **và** `s[i] == s2[j]`) là đổi chỗ **kép** — sửa 2 vị trí. Ví dụ `"abcdef" → "badcfe"`: 3 cặp `(0,1), (2,3), (4,5)` đều là đổi chỗ kép → `k = 3 = mis / 2`.

### 4.4. Ký tự lặp gây khó

`s1 = "aabc"`, `s2 = "abca"`: vị trí 1 cần `'b'` (ở vị trí 2), vị trí 2 cần `'c'` (ở vị trí 3), vị trí 3 cần `'a'` — có **hai** chữ `'a'` (vị trí 0 và 1), nhưng vị trí 0 đã đúng, không nên động vào. Chọn sai ghép có thể tốn thêm bước.

In [7]:
def mismatches(s, t):
    return [i for i in range(len(s)) if s[i] != t[i]]

for a, b, exp in EXAMPLES:
    m = len(mismatches(a, b))
    print(f"{a} -> {b}: sai {m} vị trí, cận dưới {(m + 1) // 2}, đáp án {exp}")

ab -> ba: sai 2 vị trí, cận dưới 1, đáp án 1
abc -> bca: sai 3 vị trí, cận dưới 2, đáp án 2
abc -> abc: sai 0 vị trí, cận dưới 0, đáp án 0
aabc -> abca: sai 3 vị trí, cận dưới 2, đáp án 2
abac -> baca: sai 4 vị trí, cận dưới 2, đáp án 2
abcdef -> badcfe: sai 6 vị trí, cận dưới 3, đáp án 3


## 5. Cách 1 — Brute Force: BFS với mọi phép đổi chỗ

**Ý tưởng.** Từ chuỗi `s`, sinh mọi chuỗi bằng cách đổi chỗ **bất kỳ** cặp `(i, j)`. BFS tới `s2`.

In [8]:
def k_brute(s1, s2):
    n = len(s1)
    def neighbors(s):
        for i in range(n):
            for j in range(i + 1, n):
                if s[i] != s[j]:
                    yield swap(s, i, j)
    return bfs_generic(s1, s2, neighbors)

for a, b, exp in EXAMPLES:
    assert k_brute(a, b) == exp
print("Brute force BFS: OK")

Brute force BFS: OK


**Độ phức tạp.** Mỗi chuỗi có `O(n²)` hàng xóm; số chuỗi có thể đạt tới số hoán vị đa tập của `s1` — với `n = 20` và 6 chữ cái là cỡ `20! / (3!·3!·...)` ≈ `10¹²`. Chỉ chạy được với `n <= 8–9`.

**Nút thắt.** BFS thử cả những phép đổi chỗ **vô ích** (đổi hai vị trí đã đúng, hoặc làm sai thêm). Cần thu hẹp hàng xóm mà vẫn giữ đáp án tối ưu.

## 6. Bước đột phá

**Insight 1 — Chỉ sửa vị trí sai đầu tiên.** Gọi `i` = vị trí đầu tiên mà `s[i] != s2[i]`. Trong **mọi** lời giải, vị trí `i` phải được sửa vào lúc nào đó. Có thể chứng minh luôn tồn tại một lời giải tối ưu trong đó lần đổi chỗ **đầu tiên** đưa đúng chữ `s2[i]` vào vị trí `i` (đổi thứ tự các phép đổi không làm tăng số bước). → Chỉ xét hàng xóm `swap(s, i, j)` với `j > i`.

**Insight 2 — Chỉ lấy từ vị trí đang sai.** Chọn `j` sao cho `s[j] == s2[i]` (đem đúng chữ tới) **và** `s[j] != s2[j]` (không phá một vị trí đã đúng). → Số hàng xóm giảm từ `O(n²)` xuống `O(n)`, và thực tế rất ít.

**Insight 3 — Ưu tiên đổi chỗ kép.** Nếu có `j` với `s[j] == s2[i]` **và** `s[i] == s2[j]`, đổi `i ↔ j` sửa cả hai — luôn an toàn để chọn **ngay** (không cần thử nhánh khác). Đây là ý tưởng "chu trình độ dài 2 trước".

**Insight 4 — Cận dưới cho DFS/A*.** `h(s) = ceil(mis(s) / 2)` không bao giờ vượt quá số bước thật → dùng để cắt nhánh (DFS) hoặc làm heuristic chấp nhận được (A*).

**Quy tắc nếu…thì:**
- *Nếu* `s == s2` *thì* 0.
- *Nếu* tồn tại đổi chỗ kép tại `i` *thì* chỉ đi nhánh đó.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS với cắt tỉa "vị trí sai đầu tiên"

In [9]:
def k_bfs(s1, s2):
    if s1 == s2:
        return 0
    n = len(s1)
    seen = {s1}
    q = deque([s1])
    steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            s = q.popleft()
            i = 0
            while s[i] == s2[i]:
                i += 1
            for j in range(i + 1, n):
                if s[j] == s2[i] and s[j] != s2[j]:
                    t = swap(s, i, j)
                    if t == s2:
                        return steps
                    if t not in seen:
                        seen.add(t)
                        q.append(t)
    return -1

for a, b, exp in EXAMPLES:
    assert k_bfs(a, b) == exp
print("BFS cắt tỉa: OK")

BFS cắt tỉa: OK


Giải thích:

- `while s[i] == s2[i]: i += 1` tìm vị trí sai đầu tiên (chắc chắn tồn tại vì `s != s2`).
- Mọi chuỗi trong lớp `steps − 1` đều khác `s2`; sinh con và kiểm tra `t == s2` ngay để trả sớm.

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionBFS:
    def kSimilarity(self, s1: str, s2: str) -> int:
        if s1 == s2:
            return 0
        n = len(s1)
        seen = {s1}
        q = deque([s1])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                s = q.popleft()
                i = 0
                while s[i] == s2[i]:
                    i += 1
                lst = list(s)
                for j in range(i + 1, n):
                    if s[j] == s2[i] and s[j] != s2[j]:
                        lst[i], lst[j] = lst[j], lst[i]
                        t = ''.join(lst)
                        lst[i], lst[j] = lst[j], lst[i]
                        if t == s2:
                            return steps
                        if t not in seen:
                            seen.add(t)
                            q.append(t)
        return -1

for a, b, exp in EXAMPLES:
    assert SolutionBFS().kSimilarity(a, b) == exp
print("SolutionBFS: OK")

SolutionBFS: OK


**Độ phức tạp cách 2:** khó đánh giá chặt; số trạng thái bị chặn bởi số chuỗi "tiền tố đã đúng", thực tế vài nghìn đến vài chục nghìn với `n = 20`. Mỗi trạng thái `O(n)` hàng xóm, mỗi hàng xóm `O(n)` để tạo chuỗi.

### 7.2. Cách 3 — DFS / backtracking có memo + ưu tiên đổi chỗ kép

`f(s)` = số bước tối thiểu từ `s`. Tại vị trí sai đầu tiên `i`:

- Nếu có `j` đổi chỗ kép → `f(s) = 1 + f(swap(s, i, j))` (chỉ một nhánh).
- Ngược lại, thử mọi `j` hợp lệ, lấy min.

In [11]:
def k_dfs(s1, s2):
    n = len(s1)
    memo = {}

    def f(s):
        if s == s2:
            return 0
        if s in memo:
            return memo[s]
        i = 0
        while s[i] == s2[i]:
            i += 1
        cands = [j for j in range(i + 1, n) if s[j] == s2[i] and s[j] != s2[j]]
        double = [j for j in cands if s[i] == s2[j]]
        if double:
            res = 1 + f(swap(s, i, double[0]))
        else:
            res = min(1 + f(swap(s, i, j)) for j in cands)
        memo[s] = res
        return res

    return f(s1)

for a, b, exp in EXAMPLES:
    assert k_dfs(a, b) == exp
print("DFS + memo + đổi chỗ kép: OK")

DFS + memo + đổi chỗ kép: OK


Độ sâu đệ quy ≤ số vị trí sai ≤ 20 — an toàn.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionDFS:
    def kSimilarity(self, s1: str, s2: str) -> int:
        n = len(s1)
        memo = {}

        def f(s: str) -> int:
            if s == s2:
                return 0
            if s in memo:
                return memo[s]
            i = 0
            while s[i] == s2[i]:
                i += 1
            best = float('inf')
            lst = list(s)
            for j in range(i + 1, n):
                if s[j] == s2[i] and s[j] != s2[j]:
                    lst[i], lst[j] = lst[j], lst[i]
                    cand = 1 + f(''.join(lst))
                    lst[i], lst[j] = lst[j], lst[i]
                    if s[i] == s2[j]:          # đổi chỗ kép: tối ưu, dừng ngay
                        best = cand
                        break
                    best = min(best, cand)
            memo[s] = best
            return best

        return f(s1)

for a, b, exp in EXAMPLES:
    assert SolutionDFS().kSimilarity(a, b) == exp
print("SolutionDFS: OK")

SolutionDFS: OK


Lưu ý: trong bản nộp, nếu gặp đổi chỗ kép **sau** khi đã thử vài `j` thường thì vẫn đúng (ta lấy `best = cand` của nhánh kép vì nó tối ưu; các nhánh trước không thể tốt hơn — xem mục 10).

**Độ phức tạp cách 3:** số trạng thái được memo tương tự BFS; nhánh kép cắt rất mạnh.

### 7.3. Cách 4 — A* với heuristic `ceil(mis / 2)`

`g` = số bước đã đi, `h` = cận dưới. Mở rộng theo `f = g + h` nhỏ nhất. Vì `h` chấp nhận được và nhất quán (mỗi bước `h` giảm tối đa 1), lần đầu lấy ra `s2` là tối ưu.

In [13]:
def k_astar(s1, s2):
    n = len(s1)
    def h(s):
        return (sum(1 for a, b in zip(s, s2) if a != b) + 1) // 2
    g = {s1: 0}
    pq = [(h(s1), 0, s1)]
    while pq:
        fval, gs, s = heapq.heappop(pq)
        if s == s2:
            return gs
        if gs > g[s]:
            continue
        i = 0
        while s[i] == s2[i]:
            i += 1
        for j in range(i + 1, n):
            if s[j] == s2[i] and s[j] != s2[j]:
                t = swap(s, i, j)
                if gs + 1 < g.get(t, float('inf')):
                    g[t] = gs + 1
                    heapq.heappush(pq, (gs + 1 + h(t), gs + 1, t))
    return -1

for a, b, exp in EXAMPLES:
    assert k_astar(a, b) == exp
print("A*: OK")

A*: OK


Tính nhất quán: `h(s) <= 1 + h(t)` vì một lần đổi chỗ giảm `mis` tối đa 2 → `ceil(mis/2)` giảm tối đa 1.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionAStar:
    def kSimilarity(self, s1: str, s2: str) -> int:
        n = len(s1)

        def h(s):
            return (sum(a != b for a, b in zip(s, s2)) + 1) // 2

        g = {s1: 0}
        pq = [(h(s1), 0, s1)]
        while pq:
            _, gs, s = heapq.heappop(pq)
            if s == s2:
                return gs
            if gs > g[s]:
                continue
            i = 0
            while s[i] == s2[i]:
                i += 1
            lst = list(s)
            for j in range(i + 1, n):
                if s[j] == s2[i] and s[j] != s2[j]:
                    lst[i], lst[j] = lst[j], lst[i]
                    t = ''.join(lst)
                    lst[i], lst[j] = lst[j], lst[i]
                    if gs + 1 < g.get(t, 1 << 30):
                        g[t] = gs + 1
                        heapq.heappush(pq, (gs + 1 + h(t), gs + 1, t))
        return -1

for a, b, exp in EXAMPLES:
    assert SolutionAStar().kSimilarity(a, b) == exp
print("SolutionAStar: OK")

SolutionAStar: OK


**Độ phức tạp cách 4:** thường mở rộng ít trạng thái hơn BFS nhờ heuristic; thêm `log` của heap.

### 7.4. Cách 5 — Góc nhìn chu trình: `k = mis − (số chu trình tối đa)`

Rút gọn về các vị trí sai. Xây đồ thị có hướng trên 6 chữ cái: mỗi vị trí sai `i` cho một cạnh `s1[i] → s2[i]`. Mỗi đỉnh có bán bậc vào = bán bậc ra (vì anagram), nên các cạnh phân rã được thành chu trình. Một chu trình độ dài `L` cần `L − 1` lần đổi chỗ → `k = mis − C` với `C` = **số chu trình nhiều nhất** trong một phân rã. Ta tìm `C` bằng cách: lặp lấy chu trình **ngắn nhất** trước (độ dài 2, rồi 3, ...) — với bảng chữ cái 6 chữ, tham lam theo độ dài **không** luôn tối ưu, nên ta dùng tìm kiếm có memo trên **ma trận đếm cạnh 6×6**.

In [15]:
from functools import lru_cache
from itertools import permutations as _perms

def k_cycles(s1, s2):
    letters = "abcdef"
    idx = {c: i for i, c in enumerate(letters)}
    cnt = [0] * 36
    mis = 0
    for a, b in zip(s1, s2):
        if a != b:
            cnt[idx[a] * 6 + idx[b]] += 1
            mis += 1
    # liệt kê mọi chu trình đơn trên 6 đỉnh (dạng danh sách cạnh)
    cycles = []
    for L in range(2, 7):
        for combo in _perms(range(6), L):
            if combo[0] == min(combo):        # khử trùng do quay vòng
                cycles.append(tuple(combo[t] * 6 + combo[(t + 1) % L] for t in range(L)))

    @lru_cache(None)
    def best(state):
        st = list(state)
        # chọn cạnh còn lại đầu tiên; nó phải thuộc một chu trình nào đó
        e = next((k for k in range(36) if st[k]), None)
        if e is None:
            return 0
        res = 0
        for cyc in cycles:
            if e in cyc and all(st[k] for k in cyc):
                for k in cyc:
                    st[k] -= 1
                res = max(res, 1 + best(tuple(st)))
                for k in cyc:
                    st[k] += 1
        return res

    return mis - best(tuple(cnt))

for a, b, exp in EXAMPLES:
    assert k_cycles(a, b) == exp
print("Phân rã chu trình tối đa: OK")

Phân rã chu trình tối đa: OK


Ý nghĩa: chu trình `a → b → a` ứng với hai vị trí "đổi chỗ kép". Ma trận đếm có tối đa 20 cạnh, trạng thái là bộ 36 số — memo hoá hiệu quả. Đây là lời giải đẹp về mặt lý thuyết; trên LeetCode bản BFS/DFS ngắn gọn hơn.

Code nộp LeetCode cho cách 5:

In [16]:
class SolutionCycles:
    def kSimilarity(self, s1: str, s2: str) -> int:
        return k_cycles(s1, s2)

for a, b, exp in EXAMPLES:
    assert SolutionCycles().kSimilarity(a, b) == exp
print("SolutionCycles: OK")

SolutionCycles: OK


**Độ phức tạp cách 5:** số chu trình đơn có hướng trên 6 đỉnh là 409 (`15 + 40 + 90 + 144 + 120`); số trạng thái memo phụ thuộc phân bố cạnh nhưng nhỏ vì tổng cạnh ≤ 20.

## 8. TRACE từng bước

Trace cách 2 (BFS cắt tỉa) trên ví dụ vừa `"aabbcc" → "bcacab"`: in mọi cạnh được sinh ở từng bước.

In [17]:
def trace_bfs(s1, s2):
    print(f"bắt đầu: {s1}  (đích {s2}, sai {len(mismatches(s1, s2))} vị trí)")
    if s1 == s2:
        return 0
    n = len(s1)
    seen = {s1}
    layer = [s1]
    steps = 0
    while layer:
        steps += 1
        nxt = []
        for s in layer:
            i = next(k for k in range(n) if s[k] != s2[k])
            for j in range(i + 1, n):
                if s[j] == s2[i] and s[j] != s2[j]:
                    t = swap(s, i, j)
                    kind = "kép" if s[i] == s2[j] else "đơn"
                    print(f"  bước {steps}: {s} --đổi {i}↔{j} ({kind})--> {t}")
                    if t == s2:
                        print("ĐẾN ĐÍCH sau", steps, "bước")
                        return steps
                    if t not in seen:
                        seen.add(t); nxt.append(t)
        layer = nxt
    return -1

trace_bfs("aabbcc", "bcacab")

bắt đầu: aabbcc  (đích bcacab, sai 6 vị trí)
  bước 1: aabbcc --đổi 0↔2 (kép)--> baabcc
  bước 1: aabbcc --đổi 0↔3 (đơn)--> babacc
  bước 2: baabcc --đổi 1↔4 (kép)--> bcabac
  bước 2: baabcc --đổi 1↔5 (đơn)--> bcabca
  bước 2: babacc --đổi 1↔4 (kép)--> bcbaac
  bước 2: babacc --đổi 1↔5 (đơn)--> bcbaca
  bước 3: bcabac --đổi 3↔5 (kép)--> bcacab
ĐẾN ĐÍCH sau 3 bước


3

Bảng trạng thái (một nhánh tối ưu cho `"abc" → "bca"`):

| bước | chuỗi | vị trí sai đầu | cần | lấy từ `j` | loại |
|---|---|---|---|---|---|
| 0 | abc | 0 | b | 1 | đơn |
| 1 | bac | 1 | c | 2 | kép (s[1]='a' = s2[2]) |
| 2 | bca | — | — | — | xong |

Trace DFS — in cây gọi đệ quy (thụt lề theo độ sâu):

In [18]:
def trace_dfs(s1, s2):
    n = len(s1)
    memo = {}
    def f(s, depth):
        pad = "  " * depth
        if s == s2:
            print(f"{pad}{s}: đích → 0"); return 0
        if s in memo:
            print(f"{pad}{s}: memo → {memo[s]}"); return memo[s]
        i = next(k for k in range(n) if s[k] != s2[k])
        cands = [j for j in range(i + 1, n) if s[j] == s2[i] and s[j] != s2[j]]
        double = [j for j in cands if s[i] == s2[j]]
        use = double[:1] if double else cands
        print(f"{pad}{s}: sửa vị trí {i} (cần '{s2[i]}'), thử j ∈ {use}{' [kép]' if double else ''}")
        res = min(1 + f(swap(s, i, j), depth + 1) for j in use)
        memo[s] = res
        print(f"{pad}{s} → {res}")
        return res
    return f(s1, 0)

trace_dfs("aabc", "abca")

aabc: sửa vị trí 1 (cần 'b'), thử j ∈ [2]
  abac: sửa vị trí 2 (cần 'c'), thử j ∈ [3] [kép]
    abca: đích → 0
  abac → 1
aabc → 2


2

## 9. Code nộp LeetCode

Chọn cách 2 (BFS cắt tỉa): tư duy rõ ràng, tối ưu theo BFS, đủ nhanh với `n <= 20`.

In [19]:
from collections import deque
from typing import List

class Solution:
    def kSimilarity(self, s1: str, s2: str) -> int:
        if s1 == s2:
            return 0
        n = len(s1)
        seen = {s1}
        q = deque([s1])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                s = q.popleft()
                i = 0
                while s[i] == s2[i]:           # vị trí sai đầu tiên
                    i += 1
                lst = list(s)
                for j in range(i + 1, n):
                    # đem đúng chữ s2[i] tới i, chỉ lấy từ vị trí đang sai
                    if s[j] == s2[i] and s[j] != s2[j]:
                        lst[i], lst[j] = lst[j], lst[i]
                        t = ''.join(lst)
                        lst[i], lst[j] = lst[j], lst[i]
                        if t == s2:
                            return steps
                        if t not in seen:
                            seen.add(t)
                            q.append(t)
        return -1

for a, b, exp in EXAMPLES:
    assert Solution().kSimilarity(a, b) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (chuẩn hoá lời giải).** Cho chuỗi `s ≠ s2`, `i` = vị trí sai đầu tiên. Tồn tại một dãy đổi chỗ tối ưu mà phép đầu tiên là `swap(i, j)` với `j > i`, `s[j] = s2[i]`, `s[j] ≠ s2[j]`.

*Phác thảo chứng minh.* Xét phân rã chu trình như cách 5: các vị trí sai tạo đồ thị cạnh `s[p] → s2[p]`; một lời giải tối ưu ứng với một phân rã thành tối đa `C` chu trình, chi phí `mis − C`. Cạnh của vị trí `i` (`s[i] → s2[i]`) nằm trong một chu trình `Z` của phân rã tối ưu. Trong `Z`, cạnh tiếp theo xuất phát từ chữ `s2[i]` ứng với một vị trí sai `j` có `s[j] = s2[i]` (và `j > i` vì mọi vị trí `< i` đều đúng). Đổi chỗ `i ↔ j` làm vị trí `i` đúng và rút ngắn `Z` đi 1 cạnh (nếu `Z` có độ dài 2 thì `Z` biến mất, sửa cả hai). Các chu trình khác giữ nguyên → chi phí còn lại là `mis' − C'` với `mis' − C' = (mis − C) − 1`. Vậy phép đổi chỗ đó nằm trên một lời giải tối ưu.

**Hệ quả.** Đồ thị trạng thái thu gọn (chỉ các cạnh như trên) vẫn chứa một đường đi tối ưu từ `s1` tới `s2`, và mọi cạnh của nó là phép đổi chỗ hợp lệ → BFS trên đồ thị thu gọn cho đáp án đúng.

**Đổi chỗ kép.** Nếu `s[i] = s2[j]` và `s[j] = s2[i]`, cạnh `i` và `j` tạo chu trình độ dài 2 `a → b → a`. Luôn có phân rã tối ưu chứa chu trình độ dài 2 đó (tách một chu trình 2 ra không làm giảm số chu trình tối đa — lập luận trao đổi chuẩn), nên chọn nó ngay là an toàn (cách 3).

**A*.** `h = ceil(mis/2)` là cận dưới và nhất quán → A* với kiểm tra `gs > g[s]` trả về tối ưu.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS mọi phép đổi | `O(S · n²)` với `S` = số hoán vị đa tập (cực lớn) | `O(S · n)` |
| 2. BFS cắt tỉa | `O(S' · n²)`, `S'` = số trạng thái thu gọn (nhỏ) | `O(S' · n)` |
| 3. DFS + memo + kép | tương tự 2, thường ít hơn | `O(S' · n)` |
| 4. A* | ít trạng thái hơn 2 | `O(S' · n)` |
| 5. Chu trình tối đa | phụ thuộc số trạng thái ma trận 6×6 | nhỏ |

In [20]:
import random, time

def random_pair(n, rnd, alpha="abcdef"):
    s1 = ''.join(rnd.choice(alpha) for _ in range(n))
    t = list(s1); rnd.shuffle(t)
    return s1, ''.join(t)

rnd = random.Random(854)
hard = [random_pair(20, rnd) for _ in range(5)] + [("abcdefabcdefabcdefab", "fedcbafedcbafedcbaba")]
for name, fn in [("bfs", k_bfs), ("dfs", k_dfs), ("astar", k_astar), ("cycles", k_cycles),
                 ("Solution", lambda a, b: Solution().kSimilarity(a, b))]:
    t0 = time.perf_counter()
    res = [fn(a, b) for a, b in hard]
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

bfs      : [5, 8, 10, 8, 11, 10]  (5 ms)
dfs      : [5, 8, 10, 8, 11, 10]  (1 ms)
astar    : [5, 8, 10, 8, 11, 10]  (6 ms)
cycles   : [5, 8, 10, 8, 11, 10]  (6 ms)
Solution : [5, 8, 10, 8, 11, 10]  (5 ms)


## 12. Bộ test

In [21]:
methods = {
    "brute": k_brute,
    "bfs": k_bfs,
    "dfs": k_dfs,
    "astar": k_astar,
    "cycles": k_cycles,
    "SolutionBFS": lambda a, b: SolutionBFS().kSimilarity(a, b),
    "SolutionDFS": lambda a, b: SolutionDFS().kSimilarity(a, b),
    "SolutionAStar": lambda a, b: SolutionAStar().kSimilarity(a, b),
    "SolutionCycles": lambda a, b: SolutionCycles().kSimilarity(a, b),
    "Solution": lambda a, b: Solution().kSimilarity(a, b),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ("a", "a", 0),
    ("abcd", "dabc", 3),                 # một chu trình độ dài 4
    ("abab", "baba", 2),
    ("aabbccddeeff", "ffeeddccbbaa", 6),
]
for name, fn in methods.items():
    for a, b, exp in edge:
        if name == "brute" and len(a) > 8:
            continue
        assert fn(a, b) == exp, (name, a, b, fn(a, b), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed, n <= 7: đối chiếu brute force
for seed in range(400):
    r_ = random.Random(seed)
    a, b = random_pair(r_.randint(1, 7), r_, alpha="abcdef"[: r_.randint(1, 6)])
    ref = k_brute(a, b)
    for name, fn in methods.items():
        assert fn(a, b) == ref, (name, a, b)
print("400 cặp ngẫu nhiên: khớp brute force")

# 12.3 n = 12..20: các cách nhanh khớp nhau
for seed in range(25):
    r_ = random.Random(3000 + seed)
    a, b = random_pair(r_.randint(12, 20), r_)
    vals = {name: fn(a, b) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, (a, b, vals)
print("n = 12..20: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK


400 cặp ngẫu nhiên: khớp brute force
n = 12..20: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn đổi với `j` **đầu tiên** hợp lệ

**Code sai:**

In [22]:
def wrong_greedy_first(s1, s2):
    s, k = s1, 0
    n = len(s)
    while s != s2:
        i = next(p for p in range(n) if s[p] != s2[p])
        j = next(p for p in range(i + 1, n) if s[p] == s2[i] and s[p] != s2[p])   # SAI: chọn bừa
        s = swap(s, i, j); k += 1
    return k

bad = None
for seed in range(500):
    a, b = random_pair(8, random.Random(seed), "abc")
    if wrong_greedy_first(a, b) != Solution().kSimilarity(a, b):
        bad = (a, b); break
print("phản ví dụ:", bad, "| sai:", wrong_greedy_first(*bad), " đúng:", Solution().kSimilarity(*bad))

phản ví dụ: ('bbabcbbb', 'bbbbbcab') | sai: 3  đúng: 2


**Vì sao sai:** chọn `j` sai có thể "phá" một chu trình độ dài 2 tiềm năng, khiến tốn thêm bước. Phải thử mọi `j` hợp lệ (BFS/DFS), trừ khi có đổi chỗ kép.

### Lỗi 2 — Dùng công thức `n − số chu trình` như với phần tử khác nhau

**Code sai:**

In [23]:
def wrong_perm_cycles(s1, s2):
    used = [False] * len(s2)
    p = []
    for ch in s1:                                  # SAI: ghép mỗi ký tự với lần xuất hiện ĐẦU TIÊN chưa dùng
        j = next(k for k in range(len(s2)) if s2[k] == ch and not used[k])
        used[j] = True; p.append(j)
    return len(s1) - len(cycles_of_perm(p))

print("sai:", wrong_perm_cycles("aab", "baa"), " đúng:", Solution().kSimilarity("aab", "baa"))

sai: 2  đúng: 1


**Vì sao sai:** `"aab" → "baa"` chỉ cần đổi 0↔2 (1 lần), nhưng ghép chữ `'a'` thứ nhất với vị trí `'a'` đầu tiên trong đích (vị trí 1) tạo ra một chu trình độ dài 3 → 2 lần. Với ký tự lặp, có nhiều cách ghép vị trí; ghép tuỳ tiện cho ít chu trình hơn tối ưu. Phải **tối đa hoá** số chu trình (cách 5) — đó chính là phần khó của bài.

### Lỗi 3 — Cho phép lấy chữ từ vị trí **đã đúng**

**Code sai:**

In [24]:
def wrong_take_correct(s1, s2):
    n = len(s1)
    def neighbors(s):
        i = next(p for p in range(n) if s[p] != s2[p])
        for j in range(i + 1, n):
            if s[j] == s2[i]:                      # SAI (chậm): thiếu điều kiện s[j] != s2[j]
                yield swap(s, i, j)
    return bfs_generic(s1, s2, neighbors) if s1 != s2 else 0

print("kết quả vẫn đúng:", wrong_take_correct("aabc", "abca"), "nhưng nhánh thừa làm BFS phình to.")

kết quả vẫn đúng: 2 nhưng nhánh thừa làm BFS phình to.


**Vì sao sai:** không sai đáp án nhưng sinh rất nhiều trạng thái vô ích (phá vị trí đúng rồi lại phải sửa) → TLE với `n = 20`. **Code đúng:** thêm `s[j] != s2[j]`.

### Lỗi 4 — Không đánh dấu `seen` → BFS lặp lại trạng thái

Không có `seen`, cùng một chuỗi xuất hiện nhiều lần trong hàng đợi (qua các thứ tự đổi chỗ khác nhau) → bùng nổ. Luôn đánh dấu khi đẩy vào.

## 14. Mẹo & nhận dạng pattern

- **"Số lần đổi chỗ ít nhất"** → nghĩ tới **chu trình hoán vị**; nếu phần tử lặp → bài tìm kiếm (BFS/A*/DFS) hoặc tối đa hoá số chu trình.
- **Cắt tỉa chuẩn hoá:** luôn xử lý **vị trí sai đầu tiên** — giảm nhân tố rẽ nhánh từ `n²` xuống `n`.
- **Cận dưới `ceil(sai/2)`** → heuristic cho A* hoặc cắt nhánh DFS (branch & bound).
- **Đổi chỗ kép** (chu trình độ dài 2) luôn nên làm trước.
- Bài liên quan: **765 Couples Holding Hands**, **773 Sliding Puzzle**, **2471 Minimum Number of Operations to Sort a Binary Tree by Level**, **1347**, **839 Similar String Groups**.

## 15. Biến thể

1. **Phần tử đôi một khác nhau**: đáp án `n − số chu trình` trong `O(n)`.
2. **Chỉ được đổi chỗ hai phần tử kề nhau**: đáp án = số nghịch thế (với ghép tham lam theo thứ tự xuất hiện) — Fenwick `O(n log n)`.
3. **In dãy đổi chỗ**: BFS lưu `parent`.
4. **Bảng chữ cái lớn, `n` lớn**: NP-khó tổng quát; dùng heuristic (đổi chỗ kép trước, rồi chu trình 3...).
5. **Chi phí đổi chỗ khác nhau theo cặp chữ**: Dijkstra/A* trên trạng thái.

Minh hoạ biến thể 1 và 2:

In [25]:
def min_swaps_distinct(a, b):
    pos = {ch: i for i, ch in enumerate(b)}
    p = [pos[ch] for ch in a]
    return len(a) - len(cycles_of_perm(p))

def min_adjacent_swaps(a, b):
    from collections import defaultdict, deque as dq
    where = defaultdict(dq)
    for i, ch in enumerate(b):
        where[ch].append(i)
    target = [where[ch].popleft() for ch in a]     # ghép theo thứ tự xuất hiện
    inv = 0
    for i in range(len(target)):
        for j in range(i + 1, len(target)):
            inv += target[i] > target[j]
    return inv

assert min_swaps_distinct("abcdef", "badcfe") == 3 == Solution().kSimilarity("abcdef", "badcfe")
print("đổi kề nhau 'abc'->'bca':", min_adjacent_swaps("abc", "bca"), "| đổi bất kỳ:", Solution().kSimilarity("abc", "bca"))

đổi kề nhau 'abc'->'bca': 2 | đổi bất kỳ: 2


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi phép đổi | đồ thị đầy đủ | rất lớn | rất lớn | đối chiếu, `n ≤ 8` |
| 2. BFS cắt tỉa | sửa vị trí sai đầu, lấy từ vị trí sai | nhỏ thực tế | nhỏ | **nộp bài** |
| 3. DFS + memo + kép | ưu tiên chu trình 2 | nhỏ hơn 2 | nhỏ | khi muốn cắt tỉa mạnh |
| 4. A* | `h = ceil(sai/2)` | ít trạng thái | nhỏ | bài tìm kiếm có cận dưới tốt |
| 5. Chu trình tối đa | ma trận 6×6, memo | nhỏ | nhỏ | hiểu bản chất, bảng chữ cái nhỏ |

## 17. Bài tập

**Bài 1 (đoán output).** `kSimilarity("abcd", "dcba")`? `("abcd", "bcda")`? `("aaab", "baaa")`?

**Bài 2 (tính tay).** Với `"abac" → "baca"`, liệt kê các nhánh BFS ở bước 1 và chỉ ra nhánh dẫn tới đáp án 2.

**Bài 3 (điền TODO).** Viết `k_path(s1, s2)` trả về danh sách các cặp `(i, j)` đã đổi chỗ (độ dài tối thiểu).

**Bài 4 (tự viết lại).** Viết lại cách 3 bằng **branch & bound**: DFS với biến `best` toàn cục và cắt khi `depth + ceil(mis/2) >= best`.

**Bài 5 (mini challenge).** Biến thể 2 (chỉ đổi chỗ kề nhau): chứng minh ghép tham lam theo thứ tự xuất hiện là tối ưu và cài bằng Fenwick tree.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def k_path_todo(s1, s2):
    # TODO: BFS cắt tỉa + parent[t] = (s, i, j); truy ngược
    pass

# Bài 4 — TODO
def k_branch_bound_todo(s1, s2):
    # TODO: best = số vị trí sai (cận trên); def dfs(s, depth): cắt nếu depth + ceil(mis/2) >= best
    pass

# Bài 5 — TODO
def min_adjacent_swaps_fenwick_todo(a, b):
    # TODO: target như min_adjacent_swaps; đếm nghịch thế bằng Fenwick từ phải sang trái
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: các chữ khác nhau → dùng chu trình.
- Mức 2: `dcba`: hai chu trình độ dài 2 → 2; `bcda`: một chu trình độ dài 4 → 3.
- Mức 3: `aaab → baaa`: đổi 0↔3 → 1.

**Bài 2.**
- Mức 1: vị trí sai đầu tiên là 0, cần `'b'`.
- Mức 2: `'b'` chỉ ở vị trí 1 → nhánh duy nhất `"baac"`.
- Mức 3: bước 2: vị trí 2 cần `'c'`, đổi 2↔3 (kép) → `"baca"`.

**Bài 3.**
- Mức 1: lưu `parent` như mọi BFS.
- Mức 2: cạnh lưu cặp `(i, j)`.
- Mức 3: kiểm tra bằng cách áp lại các phép đổi lên `s1`.

**Bài 4.**
- Mức 1: cận trên ban đầu `best = mis` (mỗi lần đổi sửa ít nhất một vị trí nên không bao giờ cần quá `mis` lần).
- Mức 2: ưu tiên nhánh đổi chỗ kép để tìm lời giải tốt sớm.
- Mức 3: so khớp với `Solution` trên dữ liệu ngẫu nhiên.

**Bài 5.**
- Mức 1: hai chữ giống nhau không bao giờ cần vượt qua nhau.
- Mức 2: vậy lần xuất hiện thứ `t` của chữ `c` trong `a` phải tới lần xuất hiện thứ `t` trong `b`.
- Mức 3: số đổi chỗ kề nhau tối thiểu để sắp mảng = số nghịch thế.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert [Solution().kSimilarity("abcd", t) for t in ("dcba", "bcda")] == [2, 3]
assert Solution().kSimilarity("aaab", "baaa") == 1

# Bài 2
assert Solution().kSimilarity("abac", "baca") == 2

# Bài 3
def k_path(s1, s2):
    if s1 == s2:
        return []
    n = len(s1)
    parent = {s1: None}
    q = deque([s1])
    while q:
        s = q.popleft()
        i = next(p for p in range(n) if s[p] != s2[p])
        for j in range(i + 1, n):
            if s[j] == s2[i] and s[j] != s2[j]:
                t = swap(s, i, j)
                if t not in parent:
                    parent[t] = (s, i, j)
                    if t == s2:
                        path, cur = [], t
                        while parent[cur]:
                            prev, a, b = parent[cur]
                            path.append((a, b)); cur = prev
                        return path[::-1]
                    q.append(t)

for seed in range(100):
    a, b = random_pair(random.Random(seed).randint(1, 12), random.Random(seed))
    path = k_path(a, b)
    s = a
    for i, j in path:
        s = swap(s, i, j)
    assert s == b and len(path) == Solution().kSimilarity(a, b)

# Bài 4
def k_branch_bound(s1, s2):
    n = len(s1)
    best = [sum(x != y for x, y in zip(s1, s2))]
    def dfs(s, depth):
        mis = [p for p in range(n) if s[p] != s2[p]]
        if not mis:
            best[0] = min(best[0], depth); return
        if depth + (len(mis) + 1) // 2 >= best[0]:
            return
        i = mis[0]
        cands = [j for j in mis[1:] if s[j] == s2[i]]
        cands.sort(key=lambda j: s[i] != s2[j])      # nhánh kép trước
        for j in cands:
            dfs(swap(s, i, j), depth + 1)
            if s[i] == s2[j]:
                break                                # kép: không cần thử nhánh khác
    dfs(s1, 0)
    return best[0]

for seed in range(150):
    a, b = random_pair(random.Random(seed).randint(1, 14), random.Random(seed))
    assert k_branch_bound(a, b) == Solution().kSimilarity(a, b)

# Bài 5
def min_adjacent_swaps_fenwick(a, b):
    from collections import defaultdict, deque as dq
    where = defaultdict(dq)
    for i, ch in enumerate(b):
        where[ch].append(i)
    target = [where[ch].popleft() for ch in a]
    n = len(target); tree = [0] * (n + 1); inv = 0
    for v in reversed(target):
        i = v                                         # đếm số phần tử < v đã thấy (bên phải)
        while i > 0:
            inv += tree[i]; i -= i & -i
        i = v + 1
        while i <= n:
            tree[i] += 1; i += i & -i
    return inv

for seed in range(100):
    a, b = random_pair(random.Random(seed).randint(1, 15), random.Random(seed))
    assert min_adjacent_swaps_fenwick(a, b) == min_adjacent_swaps(a, b)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (BFS cắt tỉa):**

```python
if s1 == s2: return 0
seen = {s1}; q = deque([s1]); steps = 0
while q:
    steps += 1
    for _ in range(len(q)):
        s = q.popleft()
        i = first index with s[i] != s2[i]
        for j in range(i + 1, n):
            if s[j] == s2[i] and s[j] != s2[j]:
                t = swap(s, i, j)
                if t == s2: return steps
                if t not in seen: seen.add(t); q.append(t)
```

**Checklist trước khi submit:**

- [ ] `s1 == s2` → 0.
- [ ] Chỉ sửa **vị trí sai đầu tiên** `i`.
- [ ] Chỉ đổi với `j > i` thoả `s[j] == s2[i]` **và** `s[j] != s2[j]`.
- [ ] Không tham lam một nhánh (trừ đổi chỗ kép).
- [ ] Đánh dấu `seen` khi đẩy vào hàng đợi.